### Libraries

In [3]:
import sys
import os
import json
import numpy as np
import pandas as pd

import joblib
import onnxruntime as rt

import shap

import warnings
warnings.filterwarnings('ignore')

sys.path.append('..')

### Step 1: First We Check The Existance Of ...

In [49]:
# Verify ONNX models exist
ONNX_DIR = '../models_onnx'
models   = ['cls_LightGBM.onnx', 'reg_RandomForest.onnx', 'cluster_GMM.onnx']

print(f"Checking for ONNX models in {ONNX_DIR}")
for model in models:
    model_path   = os.path.join(ONNX_DIR, model)
    exists = os.path.exists(model_path)
    size   = round(os.path.getsize(model_path) / (1024*1024), 2) if exists else 0
    if size == 0.0:
        size   = round(os.path.getsize(model_path) / (1024), 2) if exists else 0    
        print(f"   {'YES ' if exists else 'NO'} {model} ({size} KB)")
    else:
        print(f"   {'YES ' if exists else 'NO'} {model} ({size} MB)")

Checking for ONNX models in ../models_onnx
   YES  cls_LightGBM.onnx (1.05 MB)
   YES  reg_RandomForest.onnx (20.85 MB)
   YES  cluster_GMM.onnx (1.31 KB)


In [50]:
# Verify production pipelines exist
PIPE_DIR  = '../models/Production_pipelines'
pipelines = ['cls_pipeline.joblib', 'cls_fe_pipeline.joblib','reg_pipeline.joblib', 'reg_fe_pipeline.joblib']

print("\nChecking for pipelines...")
for pipe in pipelines:
    path   = os.path.join(PIPE_DIR, pipe)
    exists = os.path.exists(path)
    print(f"   {'YES ' if exists else 'NO'} {pipe}")


Checking for pipelines...
   YES  cls_pipeline.joblib
   YES  cls_fe_pipeline.joblib
   YES  reg_pipeline.joblib
   YES  reg_fe_pipeline.joblib


In [51]:
# Verify feature lists exist
feat_path = '../models/Project_Parameter_Files/feature_lists.json'
print(f"\nChecking for feature lists")
print(f"   {'YES ' if os.path.exists(feat_path) else 'NO'} feature_lists.json")


Checking for feature lists
   YES  feature_lists.json


### Step 2: Load Models + Pipelines

In [52]:
# Paths
MODEL_DIR = '../models'
ONNX_DIR  = '../models_onnx'
DATA_DIR  = '../data/engineered_data'

# Load ONNX sessions
print("Loading ONNX sessions...")
cls_session     = rt.InferenceSession(f'{ONNX_DIR}/cls_LightGBM.onnx')
reg_session     = rt.InferenceSession(f'{ONNX_DIR}/reg_RandomForest.onnx')
cluster_session = rt.InferenceSession(f'{ONNX_DIR}/cluster_GMM.onnx')
print(" DONE -->  cls_LightGBM.onnx")
print(" DONE -->  reg_RandomForest.onnx")
print(" DONE -->  cluster_GMM.onnx")

# Load preprocessing pipelines
print("\nLoading preprocessing pipelines...")
cls_pipeline = joblib.load(f'{MODEL_DIR}/Production_pipelines/cls_pipeline.joblib')
reg_pipeline = joblib.load(f'{MODEL_DIR}/Production_pipelines/reg_pipeline.joblib')
print(" DONE --> cls_pipeline.joblib")
print(" DONE --> reg_pipeline.joblib")

# Load feature engineering pipelines
print("\nLoading feature engineering pipelines...")
cls_fe_pipeline = joblib.load(f'{MODEL_DIR}/Production_pipelines/cls_fe_pipeline.joblib')
reg_fe_pipeline = joblib.load(f'{MODEL_DIR}/Production_pipelines/reg_fe_pipeline.joblib')
print(" DONE --> cls_fe_pipeline.joblib")
print(" DONE --> reg_fe_pipeline.joblib")

# Load feature lists
print("\nLoading feature lists...")
with open(f'{MODEL_DIR}/Project_Parameter_Files/feature_lists.json', 'r') as f:
    feature_lists = json.load(f)
cls_features     = feature_lists['classification']['features'] # ALso the feature for clustering
reg_features     = feature_lists['regression']['features']
print(f" DONE --> cls features : {len(cls_features)} features")
print(f" DONE --> reg features : {len(reg_features)} features")

# Load SHAP background data
print("\nLoading SHAP background data...")
X_train_cls = pd.read_csv(f'{DATA_DIR}/X_train_cls_engineered.csv')
X_train_reg = pd.read_csv(f'{DATA_DIR}/X_train_reg_engineered.csv')
print(f" DONE --> X_train_cls : {X_train_cls.shape}")
print(f" DONE --> X_train_reg : {X_train_reg.shape}")

# Load final models for SHAP
print("\nLoading models for SHAP...")
cls_model = joblib.load(f'{MODEL_DIR}/Final_Models/cls_LightGBM.joblib') # used orignal model for explanation to avoid complexity of making onnx session
reg_model = joblib.load(f'{MODEL_DIR}/Final_Models/reg_RandomForest.joblib')
print(" DONE --> cls_LightGBM.joblib")
print(" DONE --> reg_RandomForest.joblib")

Loading ONNX sessions...
 DONE -->  cls_LightGBM.onnx
 DONE -->  reg_RandomForest.onnx
 DONE -->  cluster_GMM.onnx

Loading preprocessing pipelines...
 DONE --> cls_pipeline.joblib
 DONE --> reg_pipeline.joblib

Loading feature engineering pipelines...
 DONE --> cls_fe_pipeline.joblib
 DONE --> reg_fe_pipeline.joblib

Loading feature lists...
 DONE --> cls features : 23 features
 DONE --> reg features : 16 features

Loading SHAP background data...
 DONE --> X_train_cls : (27577, 23)
 DONE --> X_train_reg : (24943, 16)

Loading models for SHAP...
 DONE --> cls_LightGBM.joblib
 DONE --> reg_RandomForest.joblib


### Step 3: Define constants

In [53]:
BEST_THRESHOLD_CLS = 0.6 # for classifcation

# for clustering
CLUSTER_LABELS = {
    0: 'High Value Borrower',
    1: 'Standard Borrower'
}

# you will find these cols from the preprocessing_pipeline.py where you run ColumnTransformer (num -> ord -> ohe -> remainder)
# Column order after ColumnTransformer output

# (ColumnTransformer adds prefixes like num__, ohe__, ord__)
CLS_PREPROCESSED_COLS = [
    'num__person_age', 'num__person_income', 'num__person_emp_length',
    'num__loan_amnt', 'num__loan_int_rate', 'num__loan_percent_income',
    'ord__loan_grade',
    'ohe__person_home_ownership_MORTGAGE', 'ohe__person_home_ownership_OWN',
    'ohe__person_home_ownership_RENT',
    'ohe__loan_intent_DEBTCONSOLIDATION', 'ohe__loan_intent_EDUCATION',
    'ohe__loan_intent_HOMEIMPROVEMENT', 'ohe__loan_intent_MEDICAL',
    'ohe__loan_intent_PERSONAL', 'ohe__loan_intent_VENTURE',
    'remainder__cb_person_default_on_file'
]

REG_PREPROCESSED_COLS = [
    'num__person_age', 'num__person_income', 'num__person_emp_length',
    'num__loan_amnt', 'num__loan_percent_income',
    'ord__loan_grade',
    'ohe__person_home_ownership_MORTGAGE', 'ohe__person_home_ownership_OWN',
    'ohe__person_home_ownership_RENT',
    'ohe__loan_intent_DEBTCONSOLIDATION', 'ohe__loan_intent_EDUCATION',
    'ohe__loan_intent_HOMEIMPROVEMENT', 'ohe__loan_intent_MEDICAL',
    'ohe__loan_intent_PERSONAL', 'ohe__loan_intent_VENTURE',
    'remainder__loan_status', 'remainder__cb_person_default_on_file'
]


# Now before passing these cols to fe pipeline we must strip the prefix to avoid conflict in cols names


# Column order the FE pipeline was fitted on
# (must match CSV column order from preprocessing output) --> match the sequence of processed_data.csv
CLS_FE_INPUT_COLS = [
    'person_age', 'person_income', 'person_emp_length', 'loan_grade',
    'loan_amnt', 'loan_int_rate', 'loan_percent_income',
    'cb_person_default_on_file',
    'person_home_ownership_MORTGAGE', 'person_home_ownership_OWN',
    'person_home_ownership_RENT',
    'loan_intent_DEBTCONSOLIDATION', 'loan_intent_EDUCATION',
    'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL',
    'loan_intent_PERSONAL', 'loan_intent_VENTURE'
]

REG_FE_INPUT_COLS = [
    'person_age', 'person_income', 'person_emp_length', 'loan_grade',
    'loan_amnt', 'loan_status', 'loan_percent_income',
    'cb_person_default_on_file',
    'person_home_ownership_MORTGAGE', 'person_home_ownership_OWN',
    'person_home_ownership_RENT',
    'loan_intent_DEBTCONSOLIDATION', 'loan_intent_EDUCATION',
    'loan_intent_HOMEIMPROVEMENT', 'loan_intent_MEDICAL',
    'loan_intent_PERSONAL', 'loan_intent_VENTURE'
]

print("Constants defined")
print(f"   BEST_THRESHOLD_CLS     : {BEST_THRESHOLD_CLS}")
print(f"   CLUSTER_LABELS         : {CLUSTER_LABELS}")
print(f"   CLS_PREPROCESSED_COLS  : {len(CLS_PREPROCESSED_COLS)} cols")
print(f"   REG_PREPROCESSED_COLS  : {len(REG_PREPROCESSED_COLS)} cols")
print(f"   CLS_FE_INPUT_COLS      : {len(CLS_FE_INPUT_COLS)} cols")
print(f"   REG_FE_INPUT_COLS      : {len(REG_FE_INPUT_COLS)} cols")

Constants defined
   BEST_THRESHOLD_CLS     : 0.6
   CLUSTER_LABELS         : {0: 'High Value Borrower', 1: 'Standard Borrower'}
   CLS_PREPROCESSED_COLS  : 17 cols
   REG_PREPROCESSED_COLS  : 17 cols
   CLS_FE_INPUT_COLS      : 17 cols
   REG_FE_INPUT_COLS      : 17 cols


### Step 4: See the Column Transformation For CLS

In [54]:
# Raw data points
sample_cls = {
    'loan_amnt'                 : 15000.0,
    'loan_grade'                : 'F',
    'loan_intent'               : 'DEBTCONSOLIDATION',
    'loan_percent_income'       : 0.43,
    'loan_int_rate'             : 18.78,
    'person_income'             : 35000.0,
    'person_age'                : 28,
    'person_emp_length'         : 2.0,
    'person_home_ownership'     : 'RENT',
    'cb_person_default_on_file' : 'N',
    'cb_person_cred_hist_length': 0.0,   # required by CrossColumnFixer
    'loan_status'               : 0
}

# Step 1 -- raw data points
df = pd.DataFrame([sample_cls])
print(f"Step 1 — Raw input shape     : {df.shape}")
print(f"         Columns : {df.columns.tolist()}")

# Step 2 — preprocess
X_pre = cls_pipeline.transform(df) # retrue numpy array
X_pre = pd.DataFrame(X_pre, columns=CLS_PREPROCESSED_COLS) # used col name to map those values
print(f"\nStep 2 — After preprocessing : {X_pre.shape}")
print(f"         Columns : {X_pre.columns.tolist()}")

# Step 3 — strip prefixes
X_pre.columns = [
    col.split('__')[1] if '__' in col else col
    for col in X_pre.columns
]
print(f"\nStep 3 — After stripping prefixes : {X_pre.shape}")
print(f"         Columns : {X_pre.columns.tolist()}")

# Step 4 — reorder
X_pre = X_pre[CLS_FE_INPUT_COLS] # Give me these columns in exactly this order
print(f"\nStep 4 — After reordering : {X_pre.shape}")
print(f"         Columns : {X_pre.columns.tolist()}")

# Step 5 — feature engineering
X_fe = cls_fe_pipeline.transform(X_pre)
X_fe = pd.DataFrame(X_fe, columns=cls_features) # that we save in feature list
print(f"\nStep 5 — After FE pipeline : {X_fe.shape}")
print(f"         Columns : {X_fe.columns.tolist()}")

# Step 6 — already correct shape, no selection needed
print(f"\n Final shape ready for ONNX : {X_fe.shape}")
print(f"   Expected  : (1, 23)")

Step 1 — Raw input shape     : (1, 12)
         Columns : ['loan_amnt', 'loan_grade', 'loan_intent', 'loan_percent_income', 'loan_int_rate', 'person_income', 'person_age', 'person_emp_length', 'person_home_ownership', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'loan_status']

Step 2 — After preprocessing : (1, 17)
         Columns : ['num__person_age', 'num__person_income', 'num__person_emp_length', 'num__loan_amnt', 'num__loan_int_rate', 'num__loan_percent_income', 'ord__loan_grade', 'ohe__person_home_ownership_MORTGAGE', 'ohe__person_home_ownership_OWN', 'ohe__person_home_ownership_RENT', 'ohe__loan_intent_DEBTCONSOLIDATION', 'ohe__loan_intent_EDUCATION', 'ohe__loan_intent_HOMEIMPROVEMENT', 'ohe__loan_intent_MEDICAL', 'ohe__loan_intent_PERSONAL', 'ohe__loan_intent_VENTURE', 'remainder__cb_person_default_on_file']

Step 3 — After stripping prefixes : (1, 17)
         Columns : ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_int_rate', 'loan_perc

In [55]:
# Convert to ML float32 — ONNX requires onnx float32 --> Because ONNX runtimes such as ONNX Runtime are optimized for 32-bit floating point math.
X_input = X_fe.values.astype(np.float32)
print(f"Input shape  : {X_input.shape}")
print(f"Input dtype  : {X_input.dtype}")

# Get input name
input_name = cls_session.get_inputs()[0].name
print(f"Input name   : {input_name}")

# Run inference
outputs = cls_session.run(None, {input_name: X_input})

# Extract probability — LightGBM returns list of dicts
prob     = float(np.array([p[1] for p in outputs[1]])[0])
decision = 'REJECTED' if prob >= BEST_THRESHOLD_CLS else 'APPROVED'

print(f"\nClassification result:")
print(f"   Default probability : {round(prob, 4)}")
print(f"   Threshold           : {BEST_THRESHOLD_CLS}")
print(f"   Decision            : {decision}")

Input shape  : (1, 23)
Input dtype  : float32
Input name   : float_input

Classification result:
   Default probability : 1.0
   Threshold           : 0.6
   Decision            : REJECTED


### Step 5: See the Column Transformation For REG

In [56]:
sample_reg = {
    'loan_amnt'                 : 15000.0,
    'loan_grade'                : 'F',
    'loan_intent'               : 'DEBTCONSOLIDATION',
    'loan_percent_income'       : 0.43,
    'person_income'             : 35000.0,
    'person_age'                : 28,
    'person_emp_length'         : 2.0,
    'person_home_ownership'     : 'RENT',
    'cb_person_default_on_file' : 'N',
    'cb_person_cred_hist_length': 0.0,
    'loan_status'               : 0      # always 0 — not yet defaulted
}

df_reg = pd.DataFrame([sample_reg])
print(f"\nStep 1 — Raw input shape : {df_reg.shape}")
print(f"         Columns : {df_reg.columns.tolist()}")

# Step 2 — preprocess
X_pre = reg_pipeline.transform(df_reg)
X_pre = pd.DataFrame(X_pre, columns=REG_PREPROCESSED_COLS)
print(f"\nStep 2 — After preprocessing : {X_pre.shape}")
print(f"         Columns : {X_pre.columns.tolist()}")

# Step 3 — strip prefixes
X_pre.columns = [
    col.split('__')[1] if '__' in col else col
    for col in X_pre.columns
]
print(f"\nStep 3 — After stripping prefixes : {X_pre.shape}")
print(f"         Columns : {X_pre.columns.tolist()}")

# Step 4 — reorder
X_pre = X_pre[REG_FE_INPUT_COLS]
print(f"\nStep 4 — After reordering : {X_pre.shape}")
print(f"         Columns : {X_pre.columns.tolist()}")

# Step 5 — feature engineering
X_fe_reg = reg_fe_pipeline.transform(X_pre)
X_fe_reg = pd.DataFrame(X_fe_reg, columns=reg_features)
print(f"\nStep 5 — After FE pipeline : {X_fe_reg.shape}")
print(f"         Columns : {X_fe_reg.columns.tolist()}")


Step 1 — Raw input shape : (1, 11)
         Columns : ['loan_amnt', 'loan_grade', 'loan_intent', 'loan_percent_income', 'person_income', 'person_age', 'person_emp_length', 'person_home_ownership', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'loan_status']

Step 2 — After preprocessing : (1, 17)
         Columns : ['num__person_age', 'num__person_income', 'num__person_emp_length', 'num__loan_amnt', 'num__loan_percent_income', 'ord__loan_grade', 'ohe__person_home_ownership_MORTGAGE', 'ohe__person_home_ownership_OWN', 'ohe__person_home_ownership_RENT', 'ohe__loan_intent_DEBTCONSOLIDATION', 'ohe__loan_intent_EDUCATION', 'ohe__loan_intent_HOMEIMPROVEMENT', 'ohe__loan_intent_MEDICAL', 'ohe__loan_intent_PERSONAL', 'ohe__loan_intent_VENTURE', 'remainder__loan_status', 'remainder__cb_person_default_on_file']

Step 3 — After stripping prefixes : (1, 17)
         Columns : ['person_age', 'person_income', 'person_emp_length', 'loan_amnt', 'loan_percent_income', 'loan_grade', 'perso

In [57]:
# Step 6 — ONNX inference
X_input    = X_fe_reg.values.astype(np.float32)
input_name = reg_session.get_inputs()[0].name

outputs    = reg_session.run(None, {input_name: X_input})

predicted_rate = round(float(outputs[0].ravel()[0]), 4)

print(f"\n Regression result:")
print(f"   Predicted interest rate : {predicted_rate}%")


 Regression result:
   Predicted interest rate : 18.7842%


### Step 6: See the Column Transformation For CLUS

In [58]:
# Run Clustering ONNX Inference
# Clustering uses the same cls features (X_fe from Step 4)

X_input    = X_fe.values.astype(np.float32)
input_name = cluster_session.get_inputs()[0].name
outputs    = cluster_session.run(None, {input_name: X_input})

print(outputs)

# Extract results
cluster_id    = int(outputs[0].ravel()[0])
cluster_label = CLUSTER_LABELS[cluster_id]
prob_high     = round(float(outputs[1][0][0]), 4)
prob_std      = round(float(outputs[1][0][1]), 4)

print(f"\nClustering result:")
print(f"   Cluster ID    : {cluster_id}")
print(f"   Cluster label : {cluster_label}")
print(f"   Probabilities :")
print(f"      High Value Borrower : {prob_high}")
print(f"      Standard Borrower   : {prob_std}")

[array([[0]], dtype=int64), array([[1., 0.]], dtype=float32)]

Clustering result:
   Cluster ID    : 0
   Cluster label : High Value Borrower
   Probabilities :
      High Value Borrower : 1.0
      Standard Borrower   : 0.0


### Step 7: SHAP Explanations for Classification

In [59]:
# Create explainer using training data as background
print("Creating SHAP explainer for classification...")
cls_explainer = shap.TreeExplainer(cls_model, X_train_cls)

# Get SHAP values for our sample
shap_values = cls_explainer.shap_values(X_fe)

# Binary classification — take class 1 (default)
if isinstance(shap_values, list):
    vals = shap_values[1][0] # true in this case
else:
    vals = shap_values[0]

# Build series for easy sorting
shap_series = pd.Series(vals, index=cls_features)

# Top 5 increasing risk
reasons_for = []
for feat, impact in shap_series.nlargest(5).items():
    reasons_for.append({
        'feature'      : feat,
        'feature_value': round(float(X_fe[feat].values[0]), 4),
        'shap_impact'  : round(float(impact), 4),
        'direction'    : 'increases default risk'
    })

# Top 5 decreasing risk
reasons_against = []
for feat, impact in shap_series.nsmallest(5).items():
    reasons_against.append({
        'feature'      : feat,
        'feature_value': round(float(X_fe[feat].values[0]), 4),
        'shap_impact'  : round(float(impact), 4),
        'direction'    : 'decreases default risk'
    })

print(f"\nSHAP Classification explanations:")
print(f"\n   Top 5 — increases default risk:")
for r in reasons_for:
    print(f"      {r['feature']:<35} shap={r['shap_impact']} ")

print(f"\n   Top 5 — decreases default risk:")
for r in reasons_against:
    print(f"      {r['feature']:<35} shap={r['shap_impact']}")

Creating SHAP explainer for classification...

SHAP Classification explanations:

   Top 5 — increases default risk:
      loan_grade_x_loan_int_rate          shap=5.873 
      loan_percent_income                 shap=3.9101 
      person_home_ownership_RENT          shap=1.3467 
      loan_int_rate                       shap=1.1621 
      loan_intent_DEBTCONSOLIDATION       shap=0.9972 

   Top 5 — decreases default risk:
      loan_intent_HOMEIMPROVEMENT         shap=-0.1754
      loan_intent_MEDICAL                 shap=-0.0537
      person_age_x_person_emp_length      shap=-0.0447
      person_income                       shap=-0.0409
      person_age                          shap=-0.0116


### Step 8: SHAP Explanations for Regression

In [60]:
print("Creating SHAP explainer for regression...")
reg_explainer = shap.TreeExplainer(reg_model, X_train_reg)

# Get SHAP values for our sample
shap_values_reg = reg_explainer.shap_values(X_fe_reg)

# Regression returns single array
if isinstance(shap_values_reg, list):
    vals_reg = shap_values_reg[0][0]
else:
    vals_reg = shap_values_reg[0]

# Build series
shap_series_reg = pd.Series(vals_reg, index=reg_features)

# Top 5 increasing rate
reasons_high = []
for feat, impact in shap_series_reg.nlargest(5).items():
    reasons_high.append({
        'feature'      : feat,
        'feature_value': round(float(X_fe_reg[feat].values[0]), 4),
        'shap_impact'  : round(float(impact), 4),
        'direction'    : 'increases interest rate'
    })

# Top 5 decreasing rate
reasons_low = []
for feat, impact in shap_series_reg.nsmallest(5).items():
    reasons_low.append({
        'feature'      : feat,
        'feature_value': round(float(X_fe_reg[feat].values[0]), 4),
        'shap_impact'  : round(float(impact), 4),
        'direction'    : 'decreases interest rate'
    })

print(f"\n SHAP Regression explanations:")
print(f"\n   Top 5 — increases interest rate:")
for r in reasons_high:
    print(f"      {r['feature']:<35} shap={r['shap_impact']}")

print(f"\n   Top 5 — decreases interest rate:")
for r in reasons_low:
    print(f"      {r['feature']:<35} shap={r['shap_impact']}")

Creating SHAP explainer for regression...

 SHAP Regression explanations:

   Top 5 — increases interest rate:
      loan_grade                          shap=3.9377
      loan_grade_x_loan_amnt              shap=1.6043
      loan_grade_x_loan_percent_income    shap=1.2485
      debt_burden_score                   shap=0.3605
      loan_amnt                           shap=0.2443

   Top 5 — decreases interest rate:
      loan_percent_income                 shap=-0.0503
      person_income                       shap=-0.0467
      income_per_emp_year                 shap=-0.0458
      person_home_ownership_OWN           shap=-0.0003
      person_age_x_person_emp_length      shap=0.0002


### Step 9: Plain English Reason Generator

In [61]:
GRADE_MAP = {
    'A': 'A (Excellent)', 'B': 'B (Good)',    'C': 'C (Fair)',
    'D': 'D (Poor)',      'E': 'E (Very Poor)','F': 'F (Bad)',
    'G': 'G (Very Bad)'
}

INTENT_MAP = {
    'DEBTCONSOLIDATION': 'debt consolidation',
    'EDUCATION'        : 'education',
    'HOMEIMPROVEMENT'  : 'home improvement',
    'MEDICAL'          : 'medical expenses',
    'PERSONAL'         : 'personal use',
    'VENTURE'          : 'business venture'
}

OWNERSHIP_MAP = {
    'RENT'    : 'renting',
    'OWN'     : 'owning',
    'MORTGAGE': 'on a mortgage for',
    'OTHER'   : 'other arrangement for'
}

def generate_plain_reasons(decision, default_prob, predicted_rate , shap_for , shap_against,
                            shap_high, shap_low,original_input, workflow):
    """
    Generate plain English reasons from SHAP + original input values.
    Args:
        decision       : APPROVED or REJECTED
        default_prob   : float 0-1
        predicted_rate : float or None
        shap_for       : top 5 features increasing default risk
        shap_against   : top 5 features decreasing default risk
        shap_high      : top 5 features increasing rate
        shap_low       : top 5 features decreasing rate
        original_input : raw user input dict
        workflow       : new_applicant or existing_loan
    Returns:
        dict: decision_reasons, rate_reasons
    """
    # Extract original values
    loan_amnt    = original_input.get('loan_amnt', 0)
    loan_grade   = original_input.get('loan_grade', '')
    loan_intent  = original_input.get('loan_intent', '')
    loan_pct_inc = original_input.get('loan_percent_income', 0)
    person_inc   = original_input.get('person_income', 0)
    emp_length   = original_input.get('person_emp_length', 0)
    home_own     = original_input.get('person_home_ownership', '')
    default_hist = original_input.get('cb_person_default_on_file', 'N')
    loan_rate    = original_input.get('loan_int_rate')

    grade_label  = GRADE_MAP.get(loan_grade, loan_grade)
    intent_label = INTENT_MAP.get(loan_intent, loan_intent.lower())
    pct_inc_disp = round(loan_pct_inc * 100, 1)
    rate_val     = loan_rate if loan_rate else predicted_rate

    # Feature name lists
    top_for     = [r['feature'] for r in shap_for]
    top_against = [r['feature'] for r in shap_against]
    top_high    = [r['feature'] for r in shap_high]
    top_low     = [r['feature'] for r in shap_low]

    decision_reasons = []
    rate_reasons     = []

    # Decision summary
    risk_pct = round(default_prob * 100, 1)
    if decision == 'REJECTED':
        decision_reasons.append(
            f"Your loan application was REJECTED with a {risk_pct}% "
            f"estimated default risk (threshold: "
            f"{round(BEST_THRESHOLD_CLS * 100)}%)."
        )
    else:
        decision_reasons.append(
            f"Your loan application was APPROVED with only a {risk_pct}% "
            f"estimated default risk (threshold: "
            f"{round(BEST_THRESHOLD_CLS * 100)}%)."
        )

    # Loan grade
    if any('loan_grade' in f for f in top_for):
        decision_reasons.append(
            f"Your loan grade {grade_label} is a high-risk grade, "
            f"which strongly increases your chance of default."
        )
    elif any('loan_grade' in f for f in top_against):
        decision_reasons.append(
            f"Your loan grade {grade_label} is a low-risk grade, "
            f"which works in your favor."
        )

    # Loan percent income
    if any('loan_percent_income' in f for f in top_for):
        decision_reasons.append(
            f"Your loan amount (${loan_amnt:,.0f}) is {pct_inc_disp}% "
            f"of your annual income (${person_inc:,.0f}), "
            f"which is very high and increases default risk."
        )
    elif any('loan_percent_income' in f for f in top_against):
        decision_reasons.append(
            f"Your loan-to-income ratio of {pct_inc_disp}% is manageable "
            f"relative to your income of ${person_inc:,.0f}."
        )

    # Interest rate
    if rate_val and any('loan_int_rate' in f for f in top_for):
        if decision == 'REJECTED':
            decision_reasons.append(
                f"Your interest rate of {round(rate_val, 2)}% is high, "
                f"which significantly increases default risk."
            )
        else:
            decision_reasons.append(
                f"Your interest rate of {round(rate_val, 2)}% has a "
                f"minor upward effect on default risk."
            )

    # Grade x rate interaction
    if any('loan_grade_x_loan_int_rate' in f for f in top_for):
        decision_reasons.append(
            f"The combination of grade {loan_grade} and a high "
            f"interest rate is the strongest signal for default risk."
        )

    # Home ownership
    if any('person_home_ownership_RENT' in f for f in top_for):
        decision_reasons.append(
            "Renting your home is associated with slightly higher "
            "default risk compared to homeowners."
        )
    elif any('person_home_ownership' in f for f in top_against):
        decision_reasons.append(
            "Owning your home or having a mortgage is a positive "
            "financial stability signal."
        )

    # Loan intent
    if any('loan_intent' in f for f in top_for):
        decision_reasons.append(
            f"Borrowing for {intent_label} is associated with "
            f"slightly higher default rates."
        )
    elif any('loan_intent' in f for f in top_against):
        decision_reasons.append(
            f"Borrowing for {intent_label} is associated with "
            f"lower default rates."
        )

    # Income 
    if any(f == 'person_income' for f in top_against):
        decision_reasons.append(
            f"Your annual income of ${person_inc:,.0f} helps "
            f"reduce default risk."
        )

    # Employment 
    if any(f == 'person_emp_length' for f in top_against):
        decision_reasons.append(
            f"Your {emp_length:.0f} years of employment history "
            f"is a positive factor."
        )

    # Rate reasons (new applicant only)
    if workflow == 'new_applicant' and predicted_rate:
        rate_reasons.append(
            f"Based on your profile, the estimated interest "
            f"rate is {round(predicted_rate, 2)}%."
        )
        if any('loan_grade' in f for f in top_high):
            rate_reasons.append(
                f"Your loan grade {grade_label} is the primary "
                f"driver of your interest rate — lower grades "
                f"receive higher rates."
            )
        if any('loan_grade_x_loan_amnt' in f for f in top_high):
            rate_reasons.append(
                f"The combination of grade {loan_grade} and loan "
                f"amount ${loan_amnt:,.0f} significantly raises "
                f"your rate."
            )
        if any('debt_burden_score' in f for f in top_high):
            rate_reasons.append(
                f"Your overall debt burden relative to income "
                f"and loan grade is high, increasing your rate."
            )
        if any(f == 'person_income' for f in top_low):
            rate_reasons.append(
                f"Your income of ${person_inc:,.0f} slightly "
                f"lowers your estimated rate."
            )

    return {
        'decision_reasons': decision_reasons,
        'rate_reasons'    : rate_reasons
    }


print(" generate_plain_reasons() defined")

 generate_plain_reasons() defined


### Step 10: Test Plain Reasons

In [62]:
plain_reasons = generate_plain_reasons(
    decision       = decision,
    default_prob   = prob,
    predicted_rate = predicted_rate,
    shap_for       = reasons_for,
    shap_against   = reasons_against,
    shap_high      = reasons_high,
    shap_low       = reasons_low,
    original_input = sample_cls,
    workflow       = 'new_applicant'
)

print(" Plain English Reasons:")
print("\n   Decision Reasons:")
for i, reason in enumerate(plain_reasons['decision_reasons'], 1):
    print(f"   {i}. {reason}")

print("\n   Rate Reasons:")
for i, reason in enumerate(plain_reasons['rate_reasons'], 1):
    print(f"   {i}. {reason}")

 Plain English Reasons:

   Decision Reasons:
   1. Your loan application was REJECTED with a 100.0% estimated default risk (threshold: 60%).
   2. Your loan grade F (Bad) is a high-risk grade, which strongly increases your chance of default.
   3. Your loan amount ($15,000) is 43.0% of your annual income ($35,000), which is very high and increases default risk.
   4. Your interest rate of 18.78% is high, which significantly increases default risk.
   5. The combination of grade F and a high interest rate is the strongest signal for default risk.
   6. Renting your home is associated with slightly higher default risk compared to homeowners.
   7. Borrowing for debt consolidation is associated with slightly higher default rates.
   8. Your annual income of $35,000 helps reduce default risk.

   Rate Reasons:
   1. Based on your profile, the estimated interest rate is 18.78%.
   2. Your loan grade F (Bad) is the primary driver of your interest rate — lower grades receive higher rates.
  

### Step 11: run_new_applicant() Full Pipeline

In [63]:
def run_new_applicant(input_data):
    """
    Full pipeline for new applicant workflow.

    Steps:
        1. Prepare regression input (no loan_int_rate)
        2. Predict interest rate via regression ONNX
        3. Inject predicted rate into input
        4. Prepare classification input
        5. Predict default risk via classification ONNX
        6. Predict cluster via clustering ONNX
        7. Compute SHAP for both models
        8. Generate plain English reasons

    Args:
        input_data: dict with 10 fields (no loan_int_rate)

    Returns:
        dict: full prediction result
    """
    # Step 1 — Regression input
    reg_input = {**input_data, 'loan_status': 0}

    df_reg = pd.DataFrame([reg_input]) # Step 1.2
    X_pre  = reg_pipeline.transform(df_reg) # Step 1.3
    X_pre  = pd.DataFrame(X_pre, columns=REG_PREPROCESSED_COLS) # Step 1.4
    X_pre.columns = [ # Step 1.5
        col.split('__')[1] if '__' in col else col
        for col in X_pre.columns
    ]
    X_pre    = X_pre[REG_FE_INPUT_COLS] # Step 1.6
    X_fe_reg = reg_fe_pipeline.transform(X_pre) # Step 1.7
    X_fe_reg = pd.DataFrame(X_fe_reg, columns=reg_features)# Step 1.8
    
    # Step 2 — Predict interest rate
    X_input        = X_fe_reg.values.astype(np.float32)
    input_name     = reg_session.get_inputs()[0].name
    outputs        = reg_session.run(None, {input_name: X_input})
    predicted_rate = round(float(outputs[0].ravel()[0]), 4)

    # Step 3 — Inject predicted rate
    cls_input = {**input_data,
                 'loan_int_rate': predicted_rate,
                 'loan_status'  : 0}

    # Step 4 — Classification input
    df_cls = pd.DataFrame([cls_input])
    X_pre  = cls_pipeline.transform(df_cls)
    X_pre  = pd.DataFrame(X_pre, columns=CLS_PREPROCESSED_COLS)
    X_pre.columns = [
        col.split('__')[1] if '__' in col else col
        for col in X_pre.columns
    ]
    X_pre   = X_pre[CLS_FE_INPUT_COLS]
    X_fe    = cls_fe_pipeline.transform(X_pre)
    X_fe    = pd.DataFrame(X_fe, columns=cls_features)

    # Step 5 — Predict default risk
    X_input    = X_fe.values.astype(np.float32)
    input_name = cls_session.get_inputs()[0].name
    outputs    = cls_session.run(None, {input_name: X_input})
    prob       = float(np.array([p[1] for p in outputs[1]])[0])
    decision   = 'REJECTED' if prob >= BEST_THRESHOLD_CLS else 'APPROVED'

    # Step 6 — Clustering
    input_name    = cluster_session.get_inputs()[0].name
    outputs       = cluster_session.run(None, {input_name: X_fe.values.astype(np.float32)})
    cluster_id    = int(outputs[0].ravel()[0])
    cluster_label = CLUSTER_LABELS[cluster_id]
    prob_high     = round(float(outputs[1][0][0]), 4)
    prob_std      = round(float(outputs[1][0][1]), 4)

    # Step 7 — SHAP
    # Classification SHAP
    shap_vals = cls_explainer.shap_values(X_fe)
    vals      = shap_vals[1][0] if isinstance(shap_vals, list) else shap_vals[0]
    shap_ser  = pd.Series(vals, index=cls_features)

    reasons_for     = [{'feature': f, 'feature_value': round(float(X_fe[f].values[0]), 4),
                         'shap_impact': round(float(v), 4),
                         'direction': 'increases default risk'}
                        for f, v in shap_ser.nlargest(5).items()]
    reasons_against = [{'feature': f, 'feature_value': round(float(X_fe[f].values[0]), 4),
                         'shap_impact': round(float(v), 4),
                         'direction': 'decreases default risk'}
                        for f, v in shap_ser.nsmallest(5).items()]

    # Regression SHAP
    shap_vals_reg = reg_explainer.shap_values(X_fe_reg)
    vals_reg      = shap_vals_reg[0][0] if isinstance(shap_vals_reg, list) else shap_vals_reg[0]
    shap_ser_reg  = pd.Series(vals_reg, index=reg_features)

    reasons_high = [{'feature': f, 'feature_value': round(float(X_fe_reg[f].values[0]), 4),
                      'shap_impact': round(float(v), 4),
                      'direction': 'increases interest rate'}
                     for f, v in shap_ser_reg.nlargest(5).items()]
    reasons_low  = [{'feature': f, 'feature_value': round(float(X_fe_reg[f].values[0]), 4),
                      'shap_impact': round(float(v), 4),
                      'direction': 'decreases interest rate'}
                     for f, v in shap_ser_reg.nsmallest(5).items()]

    # Step 8 — Plain reasons
    plain_reasons = generate_plain_reasons(
        decision       = decision,
        default_prob   = prob,
        predicted_rate = predicted_rate,
        shap_for       = reasons_for,
        shap_against   = reasons_against,
        shap_high      = reasons_high,
        shap_low       = reasons_low,
        original_input = input_data,
        workflow       = 'new_applicant'
    )

    return {
        'workflow'       : 'new_applicant',
        'prediction_data': {
            'classification': {
                'decision'               : decision,
                'default_probability'    : round(prob, 4),
                'threshold'              : BEST_THRESHOLD_CLS,
                'reasons_for_default'    : reasons_for,
                'reasons_against_default': reasons_against
            },
            'regression': {
                'predicted_interest_rate': predicted_rate,
                'reasons_high_rate'      : reasons_high,
                'reasons_low_rate'       : reasons_low
            },
            'clustering': {
                'cluster_id'   : cluster_id,
                'cluster_label': cluster_label,
                'probabilities': {
                    'High Value Borrower': prob_high,
                    'Standard Borrower'  : prob_std
                }
            },
            'plain_reasons': plain_reasons
        },
        'input_data': cls_input
    }


print(" run_new_applicant() defined")
print("""
   Steps:
   1. Prepare regression input
   2. Predict interest rate
   3. Inject predicted rate
   4. Prepare classification input
   5. Predict default risk
   6. Predict cluster
   7. SHAP for both models
   8. Plain English reasons
""")

 run_new_applicant() defined

   Steps:
   1. Prepare regression input
   2. Predict interest rate
   3. Inject predicted rate
   4. Prepare classification input
   5. Predict default risk
   6. Predict cluster
   7. SHAP for both models
   8. Plain English reasons



### Step 12: test run_new_applicant()

In [64]:
test_input = {
    'loan_amnt'                 : 15000.0,
    'loan_grade'                : 'F',
    'loan_intent'               : 'DEBTCONSOLIDATION',
    'loan_percent_income'       : 0.43,
    'person_income'             : 35000.0,
    'person_age'                : 28,
    'person_emp_length'         : 2.0,
    'person_home_ownership'     : 'RENT',
    'cb_person_default_on_file' : 'N',
    'cb_person_cred_hist_length': 3.0
}

result = run_new_applicant(test_input)

cls = result['prediction_data']['classification']
reg = result['prediction_data']['regression']
clu = result['prediction_data']['clustering']
pln = result['prediction_data']['plain_reasons']

print("run_new_applicant() test complete")
print(f"\n   Workflow   : {result['workflow']}")
print(f"\n   Classification:")
print(f"      Decision     : {cls['decision']}")
print(f"      Probability  : {cls['default_probability']}")
print(f"      Threshold    : {cls['threshold']}")
print(f"\n   Regression:")
print(f"      Predicted rate : {reg['predicted_interest_rate']}%")
print(f"\n   Clustering:")
print(f"      Cluster     : {clu['cluster_label']}")
print(f"      High Value  : {clu['probabilities']['High Value Borrower']}")
print(f"      Standard    : {clu['probabilities']['Standard Borrower']}")
print(f"\n   Plain Reasons:")
for i, r in enumerate(pln['decision_reasons'], 1):
    print(f"      {i}. {r}")

run_new_applicant() test complete

   Workflow   : new_applicant

   Classification:
      Decision     : REJECTED
      Probability  : 1.0
      Threshold    : 0.6

   Regression:
      Predicted rate : 18.7842%

   Clustering:
      Cluster     : High Value Borrower
      High Value  : 1.0
      Standard    : 0.0

   Plain Reasons:
      1. Your loan application was REJECTED with a 100.0% estimated default risk (threshold: 60%).
      2. Your loan grade F (Bad) is a high-risk grade, which strongly increases your chance of default.
      3. Your loan amount ($15,000) is 43.0% of your annual income ($35,000), which is very high and increases default risk.
      4. Your interest rate of 18.78% is high, which significantly increases default risk.
      5. The combination of grade F and a high interest rate is the strongest signal for default risk.
      6. Renting your home is associated with slightly higher default risk compared to homeowners.
      7. Borrowing for debt consolidation i

### Step 13: run_existing_loan() Full Pipeline

In [65]:
def run_existing_loan(input_data):
    """
    Full pipeline for existing loan workflow.
    Regression is skipped — loan_int_rate already provided.

    Steps:
        1. Prepare classification input
        2. Predict default risk via classification ONNX
        3. Predict cluster via clustering ONNX
        4. Compute SHAP for classification only
        5. Generate plain English reasons

    Args:
        input_data: dict with 11 fields (with loan_int_rate)

    Returns:
        dict: full prediction result
    """
    # Step 1 — Classification input
    cls_input = {**input_data, 'loan_status': 0}

    df_cls = pd.DataFrame([cls_input])
    X_pre  = cls_pipeline.transform(df_cls)
    X_pre  = pd.DataFrame(X_pre, columns=CLS_PREPROCESSED_COLS)
    X_pre.columns = [
        col.split('__')[1] if '__' in col else col
        for col in X_pre.columns
    ]
    X_pre = X_pre[CLS_FE_INPUT_COLS]
    X_fe  = cls_fe_pipeline.transform(X_pre)
    X_fe  = pd.DataFrame(X_fe, columns=cls_features)

    # Step 2 — Predict default risk
    X_input    = X_fe.values.astype(np.float32)
    input_name = cls_session.get_inputs()[0].name
    outputs    = cls_session.run(None, {input_name: X_input})
    prob       = float(np.array([p[1] for p in outputs[1]])[0])
    decision   = 'REJECTED' if prob >= BEST_THRESHOLD_CLS else 'APPROVED'

    # Step 3 — Clustering 
    input_name    = cluster_session.get_inputs()[0].name
    outputs       = cluster_session.run(None, {input_name: X_fe.values.astype(np.float32)})
    cluster_id    = int(outputs[0].ravel()[0])
    cluster_label = CLUSTER_LABELS[cluster_id]
    prob_high     = round(float(outputs[1][0][0]), 4)
    prob_std      = round(float(outputs[1][0][1]), 4)

    # Step 4 — SHAP classification only
    shap_vals = cls_explainer.shap_values(X_fe)
    vals      = shap_vals[1][0] if isinstance(shap_vals, list) else shap_vals[0]
    shap_ser  = pd.Series(vals, index=cls_features)

    reasons_for     = [{'feature': f,
                         'feature_value': round(float(X_fe[f].values[0]), 4),
                         'shap_impact'  : round(float(v), 4),
                         'direction'    : 'increases default risk'}
                        for f, v in shap_ser.nlargest(5).items()]
    reasons_against = [{'feature': f,
                         'feature_value': round(float(X_fe[f].values[0]), 4),
                         'shap_impact'  : round(float(v), 4),
                         'direction'    : 'decreases default risk'}
                        for f, v in shap_ser.nsmallest(5).items()]

    # Step 5 — Plain reasons
    plain_reasons = generate_plain_reasons(
        decision       = decision,
        default_prob   = prob,
        predicted_rate = None,
        shap_for       = reasons_for,
        shap_against   = reasons_against,
        shap_high      = [],
        shap_low       = [],
        original_input = input_data,
        workflow       = 'existing_loan'
    )

    return {
        'workflow'       : 'existing_loan',
        'prediction_data': {
            'classification': {
                'decision'               : decision,
                'default_probability'    : round(prob, 4),
                'threshold'              : BEST_THRESHOLD_CLS,
                'reasons_for_default'    : reasons_for,
                'reasons_against_default': reasons_against
            },
            'clustering': {
                'cluster_id'   : cluster_id,
                'cluster_label': cluster_label,
                'probabilities': {
                    'High Value Borrower': prob_high,
                    'Standard Borrower'  : prob_std
                }
            },
            'plain_reasons': plain_reasons
        },
        'input_data': input_data
    }


print("run_existing_loan() defined")
print("""
   Steps:
   1. Prepare classification input
   2. Predict default risk
   3. Predict cluster
   4. SHAP for classification only
   5. Plain English reasons
   Note: regression skipped — loan_int_rate already provided
""")

run_existing_loan() defined

   Steps:
   1. Prepare classification input
   2. Predict default risk
   3. Predict cluster
   4. SHAP for classification only
   5. Plain English reasons
   Note: regression skipped — loan_int_rate already provided



### Step 14: Test run_existing_loan()

In [66]:
test_input_existing = {
    'loan_amnt'                 : 15000.0,
    'loan_int_rate'             : 19.4,
    'loan_grade'                : 'F',
    'loan_intent'               : 'DEBTCONSOLIDATION',
    'loan_percent_income'       : 0.43,
    'person_income'             : 35000.0,
    'person_age'                : 28,
    'person_emp_length'         : 2.0,
    'person_home_ownership'     : 'RENT',
    'cb_person_default_on_file' : 'N',
    'cb_person_cred_hist_length': 3.0
}

result = run_existing_loan(test_input_existing)

cls = result['prediction_data']['classification']
clu = result['prediction_data']['clustering']
pln = result['prediction_data']['plain_reasons']

print(" run_existing_loan() test complete")
print(f"\n   Workflow   : {result['workflow']}")
print(f"\n   Classification:")
print(f"      Decision     : {cls['decision']}")
print(f"      Probability  : {cls['default_probability']}")
print(f"      Threshold    : {cls['threshold']}")
print(f"\n   Clustering:")
print(f"      Cluster     : {clu['cluster_label']}")
print(f"      High Value  : {clu['probabilities']['High Value Borrower']}")
print(f"      Standard    : {clu['probabilities']['Standard Borrower']}")
print(f"\n   Regression  : None (skipped)")
print(f"\n   Plain Reasons:")
for i, r in enumerate(pln['decision_reasons'], 1):
    print(f"      {i}. {r}")

 run_existing_loan() test complete

   Workflow   : existing_loan

   Classification:
      Decision     : REJECTED
      Probability  : 1.0
      Threshold    : 0.6

   Clustering:
      Cluster     : High Value Borrower
      High Value  : 1.0
      Standard    : 0.0

   Regression  : None (skipped)

   Plain Reasons:
      1. Your loan application was REJECTED with a 100.0% estimated default risk (threshold: 60%).
      2. Your loan grade F (Bad) is a high-risk grade, which strongly increases your chance of default.
      3. Your loan amount ($15,000) is 43.0% of your annual income ($35,000), which is very high and increases default risk.
      4. Your interest rate of 19.4% is high, which significantly increases default risk.
      5. The combination of grade F and a high interest rate is the strongest signal for default risk.
      6. Renting your home is associated with slightly higher default risk compared to homeowners.
      7. Borrowing for debt consolidation is associated wi

### Step 15: Test APPROVED Case

In [67]:
test_approved = {
    'loan_amnt'                 : 5000.0,
    'loan_grade'                : 'A',
    'loan_intent'               : 'EDUCATION',
    'loan_percent_income'       : 0.08,
    'person_income'             : 65000.0,
    'person_age'                : 35,
    'person_emp_length'         : 8.0,
    'person_home_ownership'     : 'MORTGAGE',
    'cb_person_default_on_file' : 'N',
    'cb_person_cred_hist_length': 10.0
}

result = run_new_applicant(test_approved)

cls = result['prediction_data']['classification']
reg = result['prediction_data']['regression']
clu = result['prediction_data']['clustering']
pln = result['prediction_data']['plain_reasons']

print("APPROVED case test complete")
print(f"\n   Workflow   : {result['workflow']}")
print(f"\n   Classification:")
print(f"      Decision     : {cls['decision']}")
print(f"      Probability  : {cls['default_probability']}")
print(f"\n   Regression:")
print(f"      Predicted rate : {reg['predicted_interest_rate']}%")
print(f"\n   Clustering:")
print(f"      Cluster     : {clu['cluster_label']}")
print(f"\n   Plain Reasons:")
for i, r in enumerate(pln['decision_reasons'], 1):
    print(f"      {i}. {r}")

APPROVED case test complete

   Workflow   : new_applicant

   Classification:
      Decision     : APPROVED
      Probability  : 0.0953

   Regression:
      Predicted rate : 10.9955%

   Clustering:
      Cluster     : Standard Borrower

   Plain Reasons:
      1. Your loan application was APPROVED with only a 9.5% estimated default risk (threshold: 60%).
      2. Your loan grade A (Excellent) is a low-risk grade, which works in your favor.
      3. Your loan-to-income ratio of 8.0% is manageable relative to your income of $65,000.
      4. Your interest rate of 11.0% has a minor upward effect on default risk.
      5. Owning your home or having a mortgage is a positive financial stability signal.
      6. Borrowing for education is associated with slightly higher default rates.
      7. Your annual income of $65,000 helps reduce default risk.


### Step 16: Get Applicant Data Through His/Her CNIC

In [4]:
sys.path.append(os.path.abspath('../database'))
from database.database import get_session, get_applicant_history

CNIC    = "42101-1234567-1"
session = get_session()

# Get applicant
applicant, predictions = get_applicant_history(session, CNIC)

if not applicant:
    print(f"No applicant found with CNIC: {CNIC}")
else:
    # Applicant info
    print("--- Applicant Info ---")
    print(f"  CNIC           : {applicant.cnic}")
    print(f"  First Seen     : {applicant.first_seen}")
    print(f"  Last Seen      : {applicant.last_seen}")
    print(f"  Total Visits   : {applicant.total_visits}")
    print(f"  Total Approved : {applicant.total_approved}")
    print(f"  Total Rejected : {applicant.total_rejected}")
    print(f"  Last Decision  : {applicant.last_decision}")

    # Last visit only
    last = predictions[0]  # most recent first
    print("\n--- Last Visit ---")
    print(f"  Prediction ID  : {last.id}")
    print(f"  Decision       : {last.decision}")
    print(f"  Default Prob   : {last.default_probability}")
    print(f"  Interest Rate  : {last.interest_rate}")
    print(f"  Cluster        : {last.cluster_label}")

    # Last visit inputs
    inp = last.inputs[0]
    print("\n--- Last Visit Inputs ---")
    print(f"  Loan Amount    : {inp.loan_amnt}")
    print(f"  Interest Rate  : {inp.loan_int_rate}")
    print(f"  Loan Grade     : {inp.loan_grade}")
    print(f"  Loan Intent    : {inp.loan_intent}")
    print(f"  Loan % Income  : {inp.loan_percent_income}")
    print(f"  Person Income  : {inp.person_income}")
    print(f"  Person Age     : {inp.person_age}")
    print(f"  Emp Length     : {inp.person_emp_length}")
    print(f"  Home Ownership : {inp.person_home_ownership}")
    print(f"  Default on File: {inp.cb_person_default_on_file}")

    # Top explanations
    exps = last.explanations
    print("\n--- Top Explanations ---")
    print(f"  {'Task':<16} {'Feature':<35} {'SHAP':>8}  Direction")
    print(f"  {'-'*16} {'-'*35} {'-'*8}  {'-'*25}")
    for exp in sorted(exps, key=lambda x: abs(x.shap_impact), reverse=True)[:10]:
        print(f"  {exp.task:<16} {exp.readable_name:<35} {exp.shap_impact:>8.4f}  {exp.direction}")

session.close()

--- Applicant Info ---
  CNIC           : 42101-1234567-1
  First Seen     : 2026-03-15 16:57:38.627692
  Last Seen      : 2026-03-16 18:32:49.595850
  Total Visits   : 7
  Total Approved : 2
  Total Rejected : 5
  Last Decision  : REJECTED

--- Last Visit ---
  Prediction ID  : 11
  Decision       : REJECTED
  Default Prob   : 1.0
  Interest Rate  : 18.7842
  Cluster        : High Value Borrower

--- Last Visit Inputs ---
  Loan Amount    : 15000.0
  Interest Rate  : 18.7842
  Loan Grade     : F
  Loan Intent    : DEBTCONSOLIDATION
  Loan % Income  : 0.43
  Person Income  : 35000.0
  Person Age     : 28
  Emp Length     : 2.0
  Home Ownership : RENT
  Default on File: N

--- Top Explanations ---
  Task             Feature                                 SHAP  Direction
  ---------------- ----------------------------------- --------  -------------------------
  classification   Loan Grade x Interest Rate            5.8730  increases default risk
  regression       Loan Grade           

### Step 17: Document report

In [68]:
# see report in docs/16_api_report.md